In [ ]:
!pip install OpenAI
!pip install openai==0.27.0

In [ ]:
import os
import openai
import pandas as pd
import time


file_path = '/content/Qualitative_coding_emotional_needs_TLR_LH (2).xlsx'
df = pd.read_excel(file_path)


openai.api_key = os.environ["OPENAI_API_KEY"]

def evaluate_emotional_support(post_content):
    messages = [
        {"role": "system", "content": "You are an AI assistant trained to evaluate whether emotional support is needed. Emotional support means offering comfort, reassurance, or empathy in response to distress."},
        {"role": "user", "content": f'''The response is: "{post_content}"

Choose the correct answer:
1. Yes, emotional support is clearly needed.
0. No, emotional support is not needed.


Only return the number representing the answer or "Not sure".'''}
    ]

    while True:
        try:
            completion = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=messages,
                max_tokens=150,
                temperature=0.7
            )
            break
        except openai.error.RateLimitError:
            print("Rate limit reached. Waiting for 10 seconds before retrying...")
            time.sleep(10)

    result = completion.choices[0].message['content'].strip()
    if result in ["0", "1"]:
        return int(result)
    else:
        return result

def get_emotional_support_reason(post_content):
    messages = [
        {"role": "system", "content": "You are an AI assistant trained to evaluate whether emotional support is needed and explain why. Emotional support means offering comfort, reassurance, or empathy in response to distress."},
        {"role": "user", "content": f'''The response is: "{post_content}"

Explain why emotional support is needed or not needed in this case.'''}
    ]

    while True:
        try:
            completion = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=messages,
                max_tokens=150,
                temperature=0.7
            )
            break
        except openai.error.RateLimitError:
            print("Rate limit reached. Waiting for 10 seconds before retrying...")
            time.sleep(10)

    reason = completion.choices[0].message['content'].strip()
    return reason


df['Emotional_Support_Needed'] = df['question_content/Original Post'].apply(evaluate_emotional_support)
df['GPT_Reason'] = df['question_content/Original Post'].apply(lambda x: get_emotional_support_reason(x) if pd.notna(x) else None)


def compare_manual_gpt(manual, gpt):
    if manual == gpt:
        return 'Match'
    else:
        return 'Mismatch: Manual says {} but GPT says {}'.format(manual, gpt)

df['Comparison'] = df.apply(lambda x: compare_manual_gpt(x['Emotional_needs'], x['Emotional_Support_Needed']) if pd.notna(x['Emotional_Support_Needed']) else 'Needs Review', axis=1)


df.to_excel('/content/reasoning2.xlsx', index=False, engine='openpyxl')

print("Evaluation completed. Results saved to '/content/reasoning2.xlsx'.")
